# Ollama Provider via Kaggle GPU + Cloudflare/ngrok

This notebook runs Ollama on Kaggle GPU resources and exposes Ollama's OpenAI-compatible API through a tunnel. Cloudflare quick tunnels are the default and require no token; ngrok is optional via `TUNNEL_PROVIDER=ngrok` and a Kaggle secret named `NGROK_AUTHTOKEN`.

In [ ]:
import os
# provider-proxy injected callback env
os.environ["OLLAMA_URL_CALLBACK"] = "https://dexter-contiguous-preimportantly.ngrok-free.dev/ollama/callback"
os.environ["OLLAMA_URL_CALLBACK_TOKEN"] = ""



In [ ]:
%%bash
set -euo pipefail

# Kaggle images can include third-party PPAs that intermittently fail DNS resolution.
# Ollama only needs zstd from Ubuntu repos, so disable PPAs before apt update.
sudo mkdir -p /etc/apt/sources.list.d/disabled
for f in /etc/apt/sources.list.d/*.list; do
  [ -e "$f" ] || continue
  if grep -qi 'ppa.launchpadcontent.net\|ppa.launchpad.net' "$f"; then
    sudo mv "$f" "/etc/apt/sources.list.d/disabled/$(basename "$f")"
  fi
done

export DEBIAN_FRONTEND=noninteractive
for attempt in 1 2 3; do
  if sudo apt-get \
    -o Acquire::Retries=3 \
    -o Acquire::http::Timeout=20 \
    -o Acquire::https::Timeout=20 \
    update -y; then
    break
  fi
  if [ "$attempt" = "3" ]; then
    exit 1
  fi
  sleep 5
done

sudo apt-get install -y --no-install-recommends zstd ca-certificates curl
curl -fsSL https://ollama.com/install.sh | sh

if ! command -v cloudflared >/dev/null 2>&1; then
  curl -fsSL -o /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
  sudo install -m 0755 /tmp/cloudflared /usr/local/bin/cloudflared
fi
pip install -q pyngrok

In [ ]:
import os

TUNNEL_PROVIDER = os.environ.get("TUNNEL_PROVIDER", "cloudflare").lower()
print(f"TUNNEL_PROVIDER={TUNNEL_PROVIDER}")

In [ ]:
import os
import subprocess
import time

MODEL = os.environ.get("OLLAMA_MODEL", "llama3.2")
os.environ["OLLAMA_HOST"] = "0.0.0.0"
os.environ["OLLAMA_ORIGINS"] = "*"
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
subprocess.run(["ollama", "pull", MODEL], check=True)

In [ ]:
import os
import re
import requests
import subprocess
import time

public_url = None
tunnel_proc = None
tunnel_provider_used = None


def stop_tunnel_proc():
    global tunnel_proc
    if tunnel_proc and tunnel_proc.poll() is None:
        tunnel_proc.terminate()
        try:
            tunnel_proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            tunnel_proc.kill()
            tunnel_proc.wait(timeout=5)
    tunnel_proc = None


def start_cloudflared():
    global public_url, tunnel_proc, tunnel_provider_used
    print("Starting Cloudflare quick tunnel...")
    tunnel_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    seen = []
    for _ in range(120):
        line = tunnel_proc.stdout.readline() if tunnel_proc.stdout else ""
        if line:
            print(line.rstrip())
            seen.append(line)
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
            if match:
                public_url = match.group(0)
                tunnel_provider_used = "cloudflare"
                return True
        if tunnel_proc.poll() is not None:
            break
        time.sleep(1)
    print("Cloudflare tunnel did not produce a URL. Output:\n" + "".join(seen[-50:]))
    return False


def start_ngrok():
    global public_url, tunnel_proc, tunnel_provider_used
    try:
        from kaggle_secrets import UserSecretsClient
        from pyngrok import ngrok

        token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
    except Exception as e:
        print(f"Unable to load NGROK_AUTHTOKEN; skipping ngrok fallback: {e}")
        return False

    if not token:
        print("Attached Kaggle secret NGROK_AUTHTOKEN is empty; skipping ngrok fallback.")
        return False
    ngrok.set_auth_token(token)

    print("Starting ngrok tunnel...")
    tunnel_proc = subprocess.Popen([
        "ngrok",
        "http",
        "11434",
        "--request-header-add",
        "ngrok-skip-browser-warning:true",
    ], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    for i in range(15):
        time.sleep(2)
        try:
            tunnels = requests.get("http://localhost:4040/api/tunnels", timeout=5).json()
            if tunnels.get("tunnels"):
                public_url = tunnels["tunnels"][0]["public_url"]
                tunnel_provider_used = "ngrok"
                return True
        except Exception as e:
            print(f"Waiting for ngrok... (attempt {i+1}/15: {e})")

    try:
        stdout, stderr = tunnel_proc.communicate(timeout=5)
    except subprocess.TimeoutExpired:
        stdout, stderr = "", "ngrok did not exit after startup timeout"
    print(f"ngrok failed.\nstdout:\n{stdout}\nstderr:\n{stderr}")
    return False


if TUNNEL_PROVIDER == "cloudflare":
    for attempt in range(1, 4):
        print(f"Cloudflare attempt {attempt}/3")
        if start_cloudflared():
            break
        stop_tunnel_proc()
        time.sleep(2)
    if not public_url:
        print("Cloudflare failed after 3 attempts. Attempting ngrok fallback...")
        if not start_ngrok():
            raise RuntimeError("Failed to start tunnel via Cloudflare and ngrok fallback.")
elif TUNNEL_PROVIDER == "ngrok":
    if not start_ngrok():
        raise RuntimeError("Failed to start ngrok tunnel.")
else:
    raise RuntimeError(f"Unknown TUNNEL_PROVIDER: {TUNNEL_PROVIDER}")

with open("ollama_base_url.txt", "w", encoding="utf-8") as f:
    f.write(public_url + "\n")
with open("ollama_provider.env", "w", encoding="utf-8") as f:
    f.write(f"OLLAMA_BASE_URL={public_url}\n")
    f.write(f"OLLAMA_MODEL={MODEL}\n")

callback_url = os.environ.get("OLLAMA_URL_CALLBACK") or os.environ.get("KAGGLE_OLLAMA_CALLBACK_URL")
callback_token = os.environ.get("OLLAMA_URL_CALLBACK_TOKEN") or os.environ.get("KAGGLE_OLLAMA_CALLBACK_TOKEN")
if callback_url:
    headers = {"authorization": f"Bearer {callback_token}"} if callback_token else {}
    payload = {"url": public_url, "model": MODEL, "provider": tunnel_provider_used or TUNNEL_PROVIDER}
    for attempt in range(1, 6):
        try:
            response = requests.post(callback_url, headers=headers, json=payload, timeout=30)
            print(f"Callback status: {response.status_code}")
            if 200 <= response.status_code < 300:
                break
            print(response.text[:500])
        except Exception as e:
            print(f"Callback failed (attempt {attempt}/5): {e}")
        time.sleep(10)

import sys
print(f"OLLAMA_BASE_URL={public_url}")
print(f"MODEL={MODEL}")
print(f"TUNNEL_PROVIDER_USED={tunnel_provider_used or TUNNEL_PROVIDER}")
print("Manifest/OpenAI-compatible base URL should point at your local proxy: http://127.0.0.1:9999/ollama/v1")
sys.stdout.flush()

In [ ]:
import requests

response = requests.post(
    "http://127.0.0.1:11434/v1/chat/completions",
    json={
        "model": MODEL,
        "stream": False,
        "messages": [{"role": "user", "content": "Reply with exactly KAGGLE_OLLAMA_OK"}],
    },
    timeout=300,
)
print(response.status_code)
print(response.text[:1000])

In [ ]:
import time

print("Provider is running. This cell keeps the notebook alive.")
print(f"Ollama tunnel: {public_url}")
print(f"Model: {MODEL}")
print("Use Ctrl+C or stop the kernel to shut down.")
while True:
    time.sleep(60)